# SwahiliNLU — Exploratory Data Analysis
> **Purpose:** Full EDA for the CIKM resource paper on a broad-domain Swahili intent dataset with tone and slot annotations.
>
> **Dataset columns expected:** `intent`, `utterance`, `slots` (JSON string), `tone`
>
> Upload your CSV to Kaggle as a dataset and update `DATA_PATH` below.

## Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Overview Statistics](#2-overview)
3. [Intent Distribution](#3-intent)
4. [Tone Distribution](#4-tone)
5. [Utterance Length Analysis](#5-length)
6. [Slot Analysis](#6-slots)
7. [Cross-tabulations (Intent × Tone)](#7-cross)
8. [Vocabulary & Linguistic Analysis](#8-vocab)
9. [Class Balance & Dataset Health](#9-health)
10. [Paper-Ready Summary Table](#10-summary)

---
## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
# Install any extras not on Kaggle by default
!pip install -q wordcloud matplotlib seaborn plotly

In [ ]:
import json
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

# ── Plotting style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F9F9F7',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'font.size': 11,
})

# Colour palette aligned with paper
TONE_COLORS   = {'imperative': '#E8593C', 'polite': '#1D9E75', 'conversational': '#378ADD'}
PALETTE_MAIN  = '#534AB7'
PALETTE_SEC   = '#1D9E75'
PALETTE_TONES = list(TONE_COLORS.values())

print('Libraries loaded ✓')

In [ ]:
# ── UPDATE THIS PATH ────────────────────────────────────────────────────────
# On Kaggle: /kaggle/input/<your-dataset-name>/your_file.csv
DATA_PATH = '/kaggle/input/swahili-intent/swahili_intent_dataset.csv'
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH)

# ── Normalise column names ───────────────────────────────────────────────────
df.columns = df.columns.str.strip().str.lower()

# ── Parse slots column (JSON string → dict) ──────────────────────────────────
def safe_parse_slots(val):
    if pd.isna(val) or val == '' or val == '{}':
        return {}
    try:
        return json.loads(val)
    except Exception:
        return {}

df['slots_dict']   = df['slots'].apply(safe_parse_slots)
df['n_slots']      = df['slots_dict'].apply(len)
df['has_slots']    = df['n_slots'] > 0
df['slot_keys']    = df['slots_dict'].apply(lambda d: list(d.keys()))

# ── Tokenisation (simple whitespace) ─────────────────────────────────────────
df['tokens']       = df['utterance'].astype(str).str.lower().str.split()
df['token_count']  = df['tokens'].apply(len)
df['char_count']   = df['utterance'].astype(str).apply(len)

# ── Tone normalise ────────────────────────────────────────────────────────────
df['tone'] = df['tone'].astype(str).str.strip().str.lower()

print(f'Dataset loaded: {len(df):,} rows × {df.shape[1]} columns')
df.head(5)

---
## 2. Overview Statistics <a id='2-overview'></a>

In [ ]:
n_utterances  = len(df)
n_intents     = df['intent'].nunique()
n_tones       = df['tone'].nunique()
n_slot_types  = len({k for keys in df['slot_keys'] for k in keys})
avg_tokens    = df['token_count'].mean()
med_tokens    = df['token_count'].median()
vocab         = {w for toks in df['tokens'] for w in toks}
n_vocab       = len(vocab)
pct_slotted   = df['has_slots'].mean() * 100

overview = pd.DataFrame({
    'Metric': [
        'Total utterances', 'Unique intents', 'Tone categories',
        'Unique slot types', 'Vocabulary size',
        'Avg tokens / utterance', 'Median tokens / utterance',
        'Utterances with ≥1 slot (%)'
    ],
    'Value': [
        f'{n_utterances:,}', n_intents, n_tones,
        n_slot_types, f'{n_vocab:,}',
        f'{avg_tokens:.1f}', f'{med_tokens:.0f}',
        f'{pct_slotted:.1f}%'
    ]
})

print('=== DATASET OVERVIEW ===')
print(overview.to_string(index=False))

---
## 3. Intent Distribution <a id='3-intent'></a>

In [ ]:
intent_counts = df['intent'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, max(5, len(intent_counts) * 0.38)))

bars = ax.barh(intent_counts.index, intent_counts.values,
               color=PALETTE_MAIN, alpha=0.85, edgecolor='white', linewidth=0.5)

# Annotate counts
for bar, val in zip(bars, intent_counts.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', ha='left', fontsize=9, color='#333')

ax.set_xlabel('Number of utterances')
ax.set_title('Utterance count per intent', fontweight='bold', pad=12)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig('fig_intent_distribution.png', bbox_inches='tight')
plt.show()
print(f'\nMost frequent intent : {intent_counts.idxmax()} ({intent_counts.max()})')
print(f'Least frequent intent: {intent_counts.idxmin()} ({intent_counts.min()})')

In [ ]:
# Pie chart — useful for paper figure
top_n = 10
top = intent_counts.sort_values(ascending=False).head(top_n)
other_count = intent_counts.sort_values(ascending=False).iloc[top_n:].sum()
if other_count > 0:
    top['other'] = other_count

fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.get_cmap('tab20')
colors = [cmap(i / len(top)) for i in range(len(top))]
wedges, texts, autotexts = ax.pie(
    top.values, labels=top.index, autopct='%1.1f%%',
    colors=colors, startangle=140, pctdistance=0.82,
    wedgeprops={'linewidth': 0.8, 'edgecolor': 'white'}
)
for at in autotexts:
    at.set_fontsize(8)
ax.set_title(f'Intent share (top {top_n})', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_intent_pie.png', bbox_inches='tight')
plt.show()

---
## 4. Tone Distribution <a id='4-tone'></a>

In [ ]:
tone_counts = df['tone'].value_counts()
tone_pct    = (tone_counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar
colors = [TONE_COLORS.get(t, '#888') for t in tone_counts.index]
axes[0].bar(tone_counts.index, tone_counts.values, color=colors, edgecolor='white', linewidth=0.8)
for i, (cnt, pct) in enumerate(zip(tone_counts.values, tone_pct.values)):
    axes[0].text(i, cnt + 1, f'{cnt}\n({pct}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Tone distribution', fontweight='bold')
axes[0].set_ylabel('Count')

# Stacked by intent (top 10 intents)
top10 = df['intent'].value_counts().head(10).index
sub   = df[df['intent'].isin(top10)]
pivot = sub.groupby(['intent', 'tone']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=['imperative', 'polite', 'conversational'], fill_value=0)
pivot.plot(kind='barh', stacked=True, ax=axes[1],
           color=[TONE_COLORS.get(c, '#888') for c in pivot.columns],
           edgecolor='white', linewidth=0.5)
axes[1].set_title('Tone per intent (top 10)', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].legend(title='Tone', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('fig_tone_distribution.png', bbox_inches='tight')
plt.show()

---
## 5. Utterance Length Analysis <a id='5-length'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Overall token distribution
axes[0].hist(df['token_count'], bins=20, color=PALETTE_MAIN, alpha=0.85, edgecolor='white')
axes[0].axvline(df['token_count'].mean(),   color='red',    linestyle='--', label=f'Mean {df["token_count"].mean():.1f}')
axes[0].axvline(df['token_count'].median(), color='orange', linestyle='--', label=f'Median {df["token_count"].median():.0f}')
axes[0].set_title('Token count distribution', fontweight='bold')
axes[0].set_xlabel('Tokens per utterance')
axes[0].legend(fontsize=9)

# Token count by tone (boxplot)
tone_order = ['imperative', 'polite', 'conversational']
tone_data  = [df[df['tone'] == t]['token_count'].dropna().values for t in tone_order]
bp = axes[1].boxplot(tone_data, labels=tone_order, patch_artist=True, notch=False,
                     medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], PALETTE_TONES):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_title('Token length by tone', fontweight='bold')
axes[1].set_ylabel('Tokens')

# Avg token count per intent (sorted)
intent_len = df.groupby('intent')['token_count'].mean().sort_values()
axes[2].barh(intent_len.index, intent_len.values, color=PALETTE_SEC, alpha=0.85, edgecolor='white')
axes[2].axvline(intent_len.mean(), color='red', linestyle='--', label=f'Overall mean {intent_len.mean():.1f}')
axes[2].set_title('Avg tokens per intent', fontweight='bold')
axes[2].set_xlabel('Avg tokens')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_utterance_length.png', bbox_inches='tight')
plt.show()

# Stats table
print('\n=== TOKEN COUNT STATS BY TONE ===')
print(df.groupby('tone')['token_count'].describe().round(2).to_string())

---
## 6. Slot Analysis <a id='6-slots'></a>

In [ ]:
# Frequency of each slot type
all_slot_keys = [k for keys in df['slot_keys'] for k in keys]
slot_freq     = pd.Series(Counter(all_slot_keys)).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Slot frequency bar
axes[0].barh(slot_freq.index[::-1], slot_freq.values[::-1],
             color=PALETTE_SEC, alpha=0.85, edgecolor='white')
for i, val in enumerate(slot_freq.values[::-1]):
    axes[0].text(val + 0.2, i, str(val), va='center', fontsize=9)
axes[0].set_title('Slot type frequency', fontweight='bold')
axes[0].set_xlabel('Occurrences')

# Slot count distribution per utterance
slot_n_dist = df['n_slots'].value_counts().sort_index()
axes[1].bar(slot_n_dist.index, slot_n_dist.values,
            color=PALETTE_MAIN, alpha=0.85, edgecolor='white')
for x, y in zip(slot_n_dist.index, slot_n_dist.values):
    axes[1].text(x, y + 0.5, str(y), ha='center', fontsize=10)
axes[1].set_title('Number of slots per utterance', fontweight='bold')
axes[1].set_xlabel('Slot count')
axes[1].set_ylabel('Utterances')
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('fig_slot_analysis.png', bbox_inches='tight')
plt.show()

print('\n=== SLOT TYPE FREQUENCY ===')
print(slot_freq.to_string())

In [ ]:
# Which slots appear in which intents?
intent_slot_map = (
    df.explode('slot_keys')
      .dropna(subset=['slot_keys'])
      .groupby(['intent', 'slot_keys'])
      .size()
      .reset_index(name='count')
)

# Heatmap: intent × slot_type
pivot_slots = intent_slot_map.pivot_table(
    index='intent', columns='slot_keys', values='count', fill_value=0
)

fig, ax = plt.subplots(figsize=(max(8, len(pivot_slots.columns) * 1.1),
                                max(5, len(pivot_slots) * 0.45)))
sns.heatmap(pivot_slots, annot=True, fmt='d', cmap='YlGnBu',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Count'})
ax.set_title('Slot type × Intent co-occurrence', fontweight='bold', pad=12)
ax.set_xlabel('Slot type')
ax.set_ylabel('Intent')
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('fig_slot_intent_heatmap.png', bbox_inches='tight')
plt.show()

---
## 7. Cross-tabulations — Intent × Tone <a id='7-cross'></a>

In [ ]:
cross = pd.crosstab(df['intent'], df['tone'])
cross_norm = cross.div(cross.sum(axis=1), axis=0)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(cross) * 0.4)))

tone_cols = [c for c in ['imperative', 'polite', 'conversational'] if c in cross_norm.columns]
cross_norm[tone_cols].plot(
    kind='barh', stacked=True, ax=axes[0],
    color=[TONE_COLORS[t] for t in tone_cols],
    edgecolor='white', linewidth=0.4
)
axes[0].set_title('Tone proportion per intent', fontweight='bold')
axes[0].set_xlabel('Proportion')
axes[0].legend(title='Tone', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Heatmap of raw counts
sns.heatmap(cross[tone_cols], annot=True, fmt='d', cmap='Purples',
            linewidths=0.5, linecolor='white', ax=axes[1],
            cbar_kws={'label': 'Count'})
axes[1].set_title('Intent × Tone count heatmap', fontweight='bold', pad=10)
axes[1].set_xlabel('Tone')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('fig_intent_tone_crosstab.png', bbox_inches='tight')
plt.show()

print('\n=== INTENT × TONE COUNTS ===')
print(cross.to_string())

---
## 8. Vocabulary & Linguistic Analysis <a id='8-vocab'></a>

In [ ]:
all_tokens = [w for toks in df['tokens'] for w in toks]
word_freq  = Counter(all_tokens)
freq_df    = pd.DataFrame(word_freq.most_common(30), columns=['word', 'freq'])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(freq_df['word'][::-1], freq_df['freq'][::-1],
        color=PALETTE_MAIN, alpha=0.85, edgecolor='white')
ax.set_title('Top 30 most frequent tokens', fontweight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig('fig_top_tokens.png', bbox_inches='tight')
plt.show()

In [ ]:
# Word cloud overall
wc = WordCloud(width=800, height=400, background_color='white',
               colormap='plasma', max_words=120, prefer_horizontal=0.8)
wc.generate_from_frequencies(word_freq)

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word cloud — full dataset', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('fig_wordcloud_all.png', bbox_inches='tight')
plt.show()

In [ ]:
# Word clouds per tone — great for paper appendix
tones = ['imperative', 'polite', 'conversational']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, tone in zip(axes, tones):
    subset = df[df['tone'] == tone]
    if len(subset) == 0:
        ax.set_visible(False)
        continue
    tokens_t  = [w for toks in subset['tokens'] for w in toks]
    freq_t    = Counter(tokens_t)
    wc_t = WordCloud(width=500, height=300, background_color='white',
                     max_words=80, prefer_horizontal=0.8,
                     colormap='viridis').generate_from_frequencies(freq_t)
    ax.imshow(wc_t, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Tone: {tone.capitalize()}\n(n={len(subset):,})', fontweight='bold')

plt.suptitle('Word clouds by tone', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_wordcloud_by_tone.png', bbox_inches='tight')
plt.show()

In [ ]:
# Hapax legomena (words appearing only once) — vocabulary richness indicator
hapax = [w for w, c in word_freq.items() if c == 1]
print(f'Vocabulary size : {len(word_freq):,}')
print(f'Hapax legomena  : {len(hapax):,}  ({len(hapax)/len(word_freq)*100:.1f}% of vocab)')
print(f'\nType-Token Ratio (TTR): {len(word_freq)/len(all_tokens):.4f}')
print(f'  (higher TTR = more lexical diversity)')

# Type-Token Ratio by tone
print('\n=== TTR BY TONE ===')
for tone in tones:
    sub   = df[df['tone'] == tone]
    toks  = [w for ts in sub['tokens'] for w in ts]
    ttr   = len(set(toks)) / len(toks) if toks else 0
    print(f'  {tone:<20} TTR = {ttr:.4f}  (vocab={len(set(toks)):,}, tokens={len(toks):,})')

---
## 9. Class Balance & Dataset Health <a id='9-health'></a>

In [ ]:
# Imbalance ratio
intent_counts = df['intent'].value_counts()
imbalance_ratio = intent_counts.max() / intent_counts.min()

print(f'Max samples in any intent : {intent_counts.max()}')
print(f'Min samples in any intent : {intent_counts.min()}')
print(f'Imbalance ratio (max/min) : {imbalance_ratio:.2f}x')
print(f'Std dev across intents    : {intent_counts.std():.1f}')

# Visualise balance with threshold line
ideal = len(df) / df['intent'].nunique()

fig, ax = plt.subplots(figsize=(10, max(4, df['intent'].nunique() * 0.38)))
colors_bar = [
    '#E8593C' if v < ideal * 0.5 else ('#EF9F27' if v < ideal * 0.75 else PALETTE_MAIN)
    for v in intent_counts.sort_values().values
]
ax.barh(intent_counts.sort_values().index,
        intent_counts.sort_values().values,
        color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(ideal, color='green', linestyle='--', linewidth=1.5, label=f'Ideal balance ({ideal:.0f})')
ax.set_title('Intent balance (red = under-represented)', fontweight='bold')
ax.set_xlabel('Utterance count')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_class_balance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Duplicate & near-duplicate check
n_exact_dup   = df.duplicated(subset=['utterance']).sum()
n_exact_dup_i = df.duplicated(subset=['utterance', 'intent']).sum()

print(f'Exact duplicate utterances              : {n_exact_dup}')
print(f'Exact duplicates (same utterance+intent): {n_exact_dup_i}')

if n_exact_dup > 0:
    print('\nSample duplicates:')
    print(df[df.duplicated(subset=['utterance'], keep=False)][['intent', 'tone', 'utterance']].head(10).to_string(index=False))

In [ ]:
# Missing values report
missing = df[['intent', 'utterance', 'slots', 'tone']].isnull().sum()
print('=== MISSING VALUES ===')
print(missing.to_string())
print(f'\nRows with any missing value: {df[["intent","utterance","tone"]].isnull().any(axis=1).sum()}')

---
## 10. Paper-Ready Summary Table <a id='10-summary'></a>

In [ ]:
# ── Per-intent summary for the paper (Table 1) ──────────────────────────────
intent_summary = (
    df.groupby('intent')
      .agg(
          utterances=('utterance', 'count'),
          avg_tokens=('token_count', 'mean'),
          avg_slots=('n_slots', 'mean'),
          slot_coverage=('has_slots', 'mean'),
      )
      .round(2)
      .reset_index()
)

# Add tone breakdown columns
tone_pivot = (
    df.groupby(['intent', 'tone'])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
intent_summary = intent_summary.merge(tone_pivot, on='intent', how='left')
intent_summary['slot_coverage'] = (intent_summary['slot_coverage'] * 100).round(1).astype(str) + '%'

print('=== PER-INTENT SUMMARY (Table 1 candidate) ===')
print(intent_summary.to_string(index=False))

# Save for paper
intent_summary.to_csv('table1_intent_summary.csv', index=False)
print('\nSaved → table1_intent_summary.csv')

In [ ]:
# ── Dataset comparison table (for related work section) ─────────────────────
comparison = pd.DataFrame([
    {'Dataset': 'ATIS',         'Language': 'English',       'Intents': 21,         'Utterances': '5,871',  'Slots': 79,  'Tone': '✗', 'Multi-domain': '✗'},
    {'Dataset': 'SNIPS',        'Language': 'English',       'Intents': 7,          'Utterances': '14,484', 'Slots': 39,  'Tone': '✗', 'Multi-domain': '✓'},
    {'Dataset': 'MASSIVE-sw',   'Language': 'Swahili',       'Intents': 60,         'Utterances': '~2,974', 'Slots': 55,  'Tone': '✗', 'Multi-domain': '✓'},
    {'Dataset': 'MultiATIS++',  'Language': '9 languages',   'Intents': 21,         'Utterances': '5,871×9','Slots': 79,  'Tone': '✗', 'Multi-domain': '✗'},
    {'Dataset': '**Ours**',     'Language': 'Swahili',       'Intents': n_intents,  'Utterances': f'{n_utterances:,}', 'Slots': n_slot_types, 'Tone': '✓', 'Multi-domain': '✓'},
])

print('=== DATASET COMPARISON TABLE (Related Work) ===')
print(comparison.to_string(index=False))
comparison.to_csv('table_dataset_comparison.csv', index=False)
print('\nSaved → table_dataset_comparison.csv')

In [ ]:
# ── Recommended train/dev/test split ─────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Stratify by intent to preserve distribution
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['intent'])
dev_df, test_df   = train_test_split(temp_df, test_size=0.667, random_state=42, stratify=temp_df['intent'])

print(f'Train : {len(train_df):>5,}  ({len(train_df)/len(df)*100:.0f}%)')
print(f'Dev   : {len(dev_df):>5,}  ({len(dev_df)/len(df)*100:.0f}%)')
print(f'Test  : {len(test_df):>5,}  ({len(test_df)/len(df)*100:.0f}%)')

# Verify stratification held
print('\nIntent distribution check (train vs test):')
dist_check = pd.DataFrame({
    'train_%': (train_df['intent'].value_counts(normalize=True) * 100).round(1),
    'test_%':  (test_df['intent'].value_counts(normalize=True) * 100).round(1),
})
dist_check['diff'] = (dist_check['train_%'] - dist_check['test_%']).abs()
print(dist_check.sort_values('diff', ascending=False).head(10).to_string())

# Save splits
train_df.to_csv('split_train.csv', index=False)
dev_df.to_csv('split_dev.csv', index=False)
test_df.to_csv('split_test.csv', index=False)
print('\nSplits saved → split_train.csv, split_dev.csv, split_test.csv')

In [ ]:
# ── Final printout for paper abstract numbers ─────────────────────────────────
print('=' * 55)
print('  NUMBERS FOR YOUR PAPER ABSTRACT / INTRODUCTION')
print('=' * 55)
print(f'  Total utterances   : {n_utterances:,}')
print(f'  Unique intents     : {n_intents}')
print(f'  Tone categories    : {n_tones} (imperative / polite / conversational)')
print(f'  Unique slot types  : {n_slot_types}')
print(f'  Vocabulary size    : {n_vocab:,} tokens')
print(f'  Avg utterance len  : {avg_tokens:.1f} tokens')
print(f'  Slot coverage      : {pct_slotted:.1f}% of utterances have ≥1 slot')
print(f'  Imbalance ratio    : {imbalance_ratio:.1f}x (max/min intent)')
print(f'  Exact duplicates   : {n_exact_dup}')
print('=' * 55)